In [1]:
import re
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, config
from tqdm.auto import tqdm

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
DEVICE

device(type='mps')

In [2]:
ds = load_dataset("imdb")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [5]:
print(ds["train"][0])
print(ds["train"][1])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [6]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

In [7]:
tokenize("Hello, my name is Egor!")

['hello', 'my', 'name', 'is', 'egor']

In [8]:
c = Counter()
c.update(["яблоко", "банан", "яблоко"])
print(c)
c.update(["персик", "банан"])
print(c)

Counter({'яблоко': 2, 'банан': 1})
Counter({'яблоко': 2, 'банан': 2, 'персик': 1})


In [9]:
counter = Counter()
for ex in tqdm(ds["train"]):
    counter.update(tokenize(ex["text"]))

  0%|          | 0/25000 [00:00<?, ?it/s]

In [10]:
dict(list(counter.items())[:15])

{'i': 87700,
 'rented': 337,
 'am': 2782,
 'curious': 262,
 'yellow': 106,
 'from': 20499,
 'my': 12505,
 'video': 1731,
 'store': 519,
 'because': 9047,
 'of': 145865,
 'all': 23987,
 'the': 336758,
 'controversy': 51,
 'that': 73287}

In [11]:
specials = {"<pad>": 0, "<unk>": 1, "<bos>": 2, "<eos>": 3}
vocab = {word: idx + len(specials) for idx, (word, _) in enumerate(counter.most_common(10000))}
vocab = {**specials, **vocab}
PAD_IDX = vocab["<pad>"]
UNK_IDX = vocab["<unk>"]
len(vocab)

10004

In [12]:
dict(list(vocab.items())[:15])

{'<pad>': 0,
 '<unk>': 1,
 '<bos>': 2,
 '<eos>': 3,
 'the': 4,
 'and': 5,
 'a': 6,
 'of': 7,
 'to': 8,
 'is': 9,
 'br': 10,
 'it': 11,
 'in': 12,
 'i': 13,
 'this': 14}

In [13]:
def encode(text, max_len=200):
    tokens = tokenize(text)
    ids = [vocab["<bos>"]] + [vocab.get(t, UNK_IDX) for t in tokens[:max_len - 2]] + [vocab["<eos>"]]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids[:max_len]


class IMDBDataset(Dataset):
    def __init__(self, data, max_len=200):
        self.data, self.labels = [], []
        for ex in tqdm(data, leave=False):
            self.data.append(encode(ex["text"], max_len))
            self.labels.append(ex["label"])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return torch.tensor(self.data[i], dtype=torch.long), torch.tensor(self.labels[i], dtype=torch.long)


train_ds = IMDBDataset(ds["train"])
test_ds = IMDBDataset(ds["test"])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)
len(train_ds), len(test_ds)

  0%|          | 0/25000 [00:00<?, ?it/s]

  0%|          | 0/25000 [00:00<?, ?it/s]

(25000, 25000)

In [14]:
ex = ds["train"][0]
ex

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [15]:
ids = encode(ex["text"])
ids[:20]

[2,
 13,
 1612,
 13,
 246,
 1997,
 4104,
 41,
 63,
 373,
 1114,
 89,
 7,
 34,
 4,
 7097,
 15,
 3400,
 11,
 57]

In [16]:
ex["label"]

0

In [17]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [18]:
torch.arange(0, 5, 2).float()

tensor([0., 2., 4.])

In [19]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(len(vocab), 128, padding_idx=PAD_IDX)
        self.pos = PositionalEncoding(128)
        layer = nn.TransformerEncoderLayer(128, nhead=4, batch_first=True, dim_feedforward=512)
        self.encoder = nn.TransformerEncoder(layer, num_layers=3)
        self.classifier = nn.Sequential(nn.Dropout(0.1), nn.Linear(128, 2))

    def forward(self, x):
        x = self.pos(self.embed(x))
        x = self.encoder(x)
        return self.classifier(x[:, 0])

In [20]:
model = Model().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [21]:
num_of_epochs = 3

for epoch in range(num_of_epochs):
    model.train()
    train_loss = 0
    progress = tqdm(train_loader, desc=f"{epoch + 1}/{num_of_epochs}")
    for i, (data, target) in enumerate(progress, 1):
        data, target = data.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(data), target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        progress.set_postfix(loss=f"{train_loss / i:.4f}")

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for data, target in tqdm(test_loader, desc="test", leave=False):
            data, target = data.to(DEVICE), target.to(DEVICE)
            preds = model(data).argmax(dim=1)
            correct += (preds == target).sum().item()
            total += target.size(0)

    print(epoch + 1, f"{train_loss / len(train_loader):.4f}", f"{correct / total:.2%}")

1/3:   0%|          | 0/782 [00:00<?, ?it/s]

test:   0%|          | 0/782 [00:00<?, ?it/s]

1 0.6284 73.92%


2/3:   0%|          | 0/782 [00:00<?, ?it/s]

test:   0%|          | 0/782 [00:00<?, ?it/s]

2 0.4566 79.98%


3/3:   0%|          | 0/782 [00:00<?, ?it/s]

test:   0%|          | 0/782 [00:00<?, ?it/s]

3 0.3697 82.57%


In [22]:
def predict(text):
    x = torch.tensor(encode(text), dtype=torch.long).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        pred = model(x).argmax().item()
    return "positive" if pred == 1 else "negative"

In [23]:
predict("Brilliant movie! Absolutely loved it!")

'positive'

In [30]:
predict("It is a very awful film!")

'negative'

In [25]:
predict("It was a waste of time!")

'negative'

In [26]:
predict("It was a most valuable spend of time!")

'positive'

In [27]:
predict("Fantastic film!")

'positive'